# VirtualDB Tutorial: SQL-First Cross-Dataset Queries

The `VirtualDB` class provides a SQL query interface across heterogeneous
Huggingface datasets stored in Parquet format. See the [VirtualDB configuration
documentation](../virtual_db_configuration.md) for more details on how to set up your
datasets for use with `VirtualDB`.

## Configuration

The configuration for `VirtualDB` is defined in a YAML file that specifies the datasets to include, their locations, and any relevant metadata or mappings. Below is an example configuration:

In [1]:
config_yaml = """
repositories:
  BrentLab/harbison_2004:
    dataset:
      harbison_2004:
        db_name: harbison
        sample_id:
          field: sample_id
        carbon_source:
          field: condition
          path: media.carbon_source.compound
        temperature_celsius:
          field: condition
          path: temperature_celsius
          dtype: numeric
        regulator_locus_tag:
          field: regulator_locus_tag
        regulator_symbol:
          field: regulator_symbol

  BrentLab/kemmeren_2014:
    dataset:
      kemmeren_2014:
        db_name: kemmeren
        sample_id:
          field: sample_id
        carbon_source:
          path: experimental_conditions.media.carbon_source.compound
        temperature_celsius:
          path: experimental_conditions.temperature_celsius
          dtype: numeric
        regulator_locus_tag:
          field: regulator_locus_tag
        regulator_symbol:
          field: regulator_symbol

  BrentLab/hackett_2020:
    dataset:
      hackett_2020:
        db_name: hackett
        sample_id:
          field: sample_id
        carbon_source:
          path: experimental_conditions.media.carbon_source.compound
        temperature_celsius:
          path: experimental_conditions.temperature_celsius
          dtype: numeric
        regulator_locus_tag:
          field: regulator_locus_tag
        regulator_symbol:
          field: regulator_symbol

  BrentLab/yeast_comparative_analysis:
    dataset:
      dto:
        dto_pvalue:
          field: dto_empirical_pvalue
        dto_fdr:
          field: dto_fdr
        links:
          binding_id:
            - [BrentLab/harbison_2004, harbison_2004]
          perturbation_id:
            - [BrentLab/kemmeren_2014, kemmeren_2014]
            - [BrentLab/hackett_2020, hackett_2020]

factor_aliases:
  carbon_source:
    glucose: [D-glucose, dextrose, glu]
    galactose: [D-galactose, gal]
    raffinose: [D-raffinose]

missing_value_labels:
  carbon_source: unspecified

description:
  carbon_source: The carbon source provided during growth
  temperature_celsius: Growth temperature in degrees Celsius
"""

import tempfile
from pathlib import Path

temp_config = Path(tempfile.mkdtemp()) / "vdb_config.yaml"
temp_config.write_text(config_yaml)
print(f"Config saved to: {temp_config}")

Config saved to: /tmp/tmp60wlr_qk/vdb_config.yaml


## Initializing VirtualDB

Creating a VirtualDB instance loads and validates the config but does
**not** download any data yet. Views are registered lazily on the first
`query()`, `tables()`, or `describe()` call.

In [2]:
from tfbpapi.virtual_db import VirtualDB

# Pass an HF token if the repos are private:
# vdb = VirtualDB(str(temp_config), token="hf_...")
vdb = VirtualDB(str(temp_config))
print(repr(vdb))

/home/chase/code/tfbp/tfbpapi/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VirtualDB(4 repos, 4 datasets, views not yet registered)


## Schema Discovery

Use `tables()`, `describe()`, `get_fields()`, and `get_common_fields()`
to explore the registered views before writing SQL.

Note that primary datasets get **two** views each:
- `<db_name>` -- the full measurement-level data (one row per sample-target pair)
- `<db_name>_meta` -- deduplicated sample-level metadata (one row per sample),
  including derived columns from config property mappings (e.g., `carbon_source` resolved from DataCard field definitions, with factor aliases applied).

In [3]:
# List all registered views
print("Registered views:")
for name in vdb.tables():
    print(f"  {name}")

Registered views:


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 4471.54it/s]
No metadata fields found for data config 'dto' in repo 'BrentLab/yeast_comparative_analysis' -- no embedded metadata_fields and no metadata config with applies_to
Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 12571.60it/s]
Key 'carbon_source' not found at path 'media.carbon_source' (current keys: ['name'])
Key 'carbon_source' not found at path 'media.carbon_source' (current keys: ['name'])
Key 'carbon_source' not found at path 'media.carbon_source' (current keys: ['name'])
Key 'temperature_celsius' not found at path 'temperature_celsius' (current keys: ['description', 'initial_temperature_celsius', 'temperature_shift_celsius', 'temperature_shift_duration_minutes', 'growth_phase_at_harvest', 'media'])


  dto_expanded
  hackett
  hackett_meta
  harbison
  harbison_meta
  kemmeren
  kemmeren_meta


In [4]:
# The _meta view has sample-level metadata plus derived columns
# (carbon_source, temperature_celsius resolved from condition definitions)
vdb.describe("harbison_meta")

,table,column_name,column_type,null,key,default,extra
0,harbison_meta,sample_id,INTEGER,YES,None,None,None
1,harbison_meta,condition,VARCHAR,YES,None,None,None
2,harbison_meta,regulator_locus_tag,VARCHAR,YES,None,None,None
3,harbison_meta,regulator_symbol,VARCHAR,YES,None,None,None
4,harbison_meta,carbon_source,VARCHAR,YES,None,None,None
5,harbison_meta,temperature_celsius,DOUBLE,YES,None,None,None


In [5]:
# The full view has measurement-level data (one row per sample-target pair)
vdb.describe("harbison")

,table,column_name,column_type,null,key,default,extra
0,harbison,sample_id,INTEGER,YES,None,None,None
1,harbison,db_id,DOUBLE,YES,None,None,None
2,harbison,regulator_locus_tag,VARCHAR,YES,None,None,None
3,harbison,regulator_symbol,VARCHAR,YES,None,None,None
4,harbison,condition,VARCHAR,YES,None,None,None
5,harbison,target_locus_tag,VARCHAR,YES,None,None,None
6,harbison,target_symbol,VARCHAR,YES,None,None,None
7,harbison,effect,DOUBLE,YES,None,None,None
8,harbison,pvalue,DOUBLE,YES,None,None,None
9,harbison,carbon_source,VARCHAR,YES,None,None,None


In [6]:
# Columns common to ALL primary dataset views
print("Common fields:", vdb.get_common_fields())

Common fields: ['carbon_source', 'regulator_locus_tag', 'regulator_symbol', 'sample_id', 'temperature_celsius']


## Querying VirtualDB

The `.query()` method executes SQL queries against the registered views. You can write complex SQL queries that join across multiple datasets, filter based on metadata, and aggregate results as needed.  

You can also use parameterized queries to safely inject variables into your SQL statements, and prepared statements for repeated queries with different parameters. 
Named prepared statements can be passed to `.prepare()` and then executed with
`.query()` with any parameterized values passed in as an arbitrary number of key/value
arguments.

In [7]:
# Query the _meta view for sample-level metadata (one row per sample)
# Note: carbon_source is derived from the condition column's DataCard definitions
# with factor aliases already applied (D-glucose -> glucose)
df_meta = vdb.query("SELECT * FROM harbison_meta LIMIT 5")
df_meta

,sample_id,condition,regulator_locus_tag,regulator_symbol,carbon_source,temperature_celsius
0,206,YPD,YKL072W,STB6,glucose,30.0
1,249,YPD,YMR016C,SOK2,glucose,30.0
2,309,YPD,YOR162C,YRR1,glucose,30.0
3,209,YPD,YKL109W,HAP4,glucose,30.0
4,189,YPD,YJR140C,HIR3,glucose,30.0


## 5. Parameterized Queries

Pass keyword arguments to `query()` and reference them with
DuckDB's `$name` syntax.

In [8]:
# A parameterized query has the following form, where `$reg` is a placeholder
# that gets replaced with the value provided in the `reg` argument.
vdb.query(
    "SELECT * FROM harbison WHERE regulator_symbol = $reg LIMIT 5",
    reg="REB1",
)

# A parameterized query can be saved for future use with the `.prepare()` method

,sample_id,db_id,regulator_locus_tag,regulator_symbol,condition,target_locus_tag,target_symbol,effect,pvalue,carbon_source,temperature_celsius
0,15,14.0,YBR049C,REB1,YPD,YPR204W,YPR204W,0.852889,0.769430,glucose,30.0
1,15,14.0,YBR049C,REB1,YPD,YPR203W,YPR203W,1.249003,0.112376,glucose,30.0
2,15,14.0,YBR049C,REB1,YPD,YPR202W,YPR202W,1.249003,0.112376,glucose,30.0
3,15,14.0,YBR049C,REB1,YPD,YPR201W,ARR3,1.513707,0.168133,glucose,30.0
4,15,14.0,YBR049C,REB1,YPD,YPR200C,ARR2,1.513707,0.168133,glucose,30.0


##  Prepared Queries

Use `prepare()` to register a named, reusable query template.
Then call it by name via `query()`.

In [9]:
# Register a prepared query
vdb.prepare("glucose_regs", """
    SELECT regulator_symbol, COUNT(*) AS n
    FROM harbison_meta
    WHERE carbon_source = $cs
    GROUP BY regulator_symbol
    HAVING n >= $min_n
    ORDER BY n DESC
""")

# note that rather than a SQL statement, we pass in the name of the prepared query
# and provide the appropriate parameters
vdb.query("glucose_regs", cs="glucose", min_n=2)

,regulator_symbol,n
0,MSN2,6
1,MSN4,5
2,RTG3,4
3,STE12,4
4,HSF1,4
...,...,...
58,GCN4,2
59,PUT3,2
60,RTG1,2
61,MOT2,2


## 7. Comparative Dataset Views

Comparative datasets (those with `links`) get an extra view type:

**`<name>_expanded`**: For each composite ID field, adds two parsed columns:
- `<link_field>_source` -- the source dataset, aliased to `db_name` when
  the `repo_id;config_name` pair is in the VirtualDB config.
- `<link_field>_id` -- the sample_id component.

This makes it easy to join or filter by source dataset without manually
parsing composite IDs.

In [10]:
# The expanded view has parsed _source and _id columns for each link field
vdb.query("SELECT * FROM dto_expanded LIMIT 3")

,binding_id,perturbation_id,binding_rank_threshold,perturbation_rank_threshold,binding_set_size,perturbation_set_size,dto_fdr,dto_empirical_pvalue,binding_repo_dataset,perturbation_repo_dataset,binding_id_id,binding_id_source,perturbation_id_id,perturbation_id_source
0,BrentLab/harbison_2004;harbison_2004;105,BrentLab/hughes_2006;overexpression;10,11.0,206.0,12.0,206.0,0.041293,0.017,harbison_2004-harbison_2004,hughes_2006-overexpression,105,harbison,10,BrentLab/hughes_2006;overexpression
1,BrentLab/harbison_2004;harbison_2004;108,BrentLab/hughes_2006;overexpression;11,60.0,67.0,60.0,67.0,0.054284,0.000,harbison_2004-harbison_2004,hughes_2006-overexpression,108,harbison,11,BrentLab/hughes_2006;overexpression
2,BrentLab/harbison_2004;harbison_2004;109,BrentLab/hughes_2006;overexpression;11,27.0,1265.0,27.0,1265.0,0.123214,0.057,harbison_2004-harbison_2004,hughes_2006-overexpression,109,harbison,11,BrentLab/hughes_2006;overexpression


In [11]:
# Join harbison metadata to dto via the expanded view's parsed columns
vdb.query("""
    SELECT h.*, d.dto_empirical_pvalue, d.dto_fdr
    FROM harbison_meta h
    JOIN dto_expanded d
        ON CAST(h.sample_id AS VARCHAR) = d.binding_id_id
        AND d.binding_id_source = 'harbison'
    WHERE d.dto_empirical_pvalue <= 0.01
    ORDER BY d.dto_empirical_pvalue
    LIMIT 10
""")

,sample_id,condition,regulator_locus_tag,regulator_symbol,carbon_source,temperature_celsius,dto_empirical_pvalue,dto_fdr
0,284,YPD,YNL216W,RAP1,glucose,30.0,0.0,0.053634
1,249,YPD,YMR016C,SOK2,glucose,30.0,0.0,0.120648
2,8,H2O2Lo,YBL103C,RTG3,glucose,30.0,0.0,0.035347
3,255,H2O2Lo,YMR037C,MSN2,glucose,30.0,0.0,0.169259
4,83,YPD,YEL009C,GCN4,glucose,30.0,0.0,0.096846
5,119,H2O2Lo,YGL073W,HSF1,glucose,30.0,0.0,0.118085
6,224,YPD,YLR131C,ACE2,glucose,30.0,0.0,0.024163
7,55,YPD,YDR146C,SWI5,glucose,30.0,0.0,0.015389
8,189,YPD,YJR140C,HIR3,glucose,30.0,0.0,0.003987
9,246,YPD,YML099C,ARG81,glucose,30.0,0.0,0.000002


In [12]:
# Cross-dataset join: harbison binding with hackett perturbation data
# via the DTO comparative dataset
vdb.query("""
    SELECT
        h.sample_id        AS harbison_sample_id,
        h.regulator_symbol,
        d.dto_empirical_pvalue,
        d.perturbation_id_id AS hackett_sample_id
    FROM harbison_meta h
    JOIN dto_expanded d
        ON CAST(h.sample_id AS VARCHAR) = d.binding_id_id
        AND d.binding_id_source = 'harbison'
    WHERE d.dto_empirical_pvalue <= 0.01
    ORDER BY d.dto_empirical_pvalue
    LIMIT 10
""")

,harbison_sample_id,regulator_symbol,dto_empirical_pvalue,hackett_sample_id
0,224,ACE2,0.0,169
1,83,GCN4,0.0,48_190
2,189,HIR3,0.0,772
3,75,CAD1,0.0,360
4,284,RAP1,0.0,96_238
5,283,RAP1,0.0,96_238
6,8,RTG3,0.0,109_251
7,246,ARG81,0.0,187
8,55,SWI5,0.0,128_270
9,119,HSF1,0.0,88


## A realistic example

Hackett has multiple experimental conditions that are unique to that dataset. There are
some regulators which have replicates within those conditions. We need to find those 
regulators and design a query which returns only 1 sample per condition set.

In [13]:
# Query hackett to find regulators with multiple samples in the same (time, mechanism)
# condition
vdb.query("""
    SELECT regulator_symbol, time, mechanism, restriction, COUNT(*) AS n
    FROM hackett_meta
    GROUP BY regulator_symbol, time, mechanism, restriction
    HAVING n > 1
    ORDER BY n DESC
""")

,regulator_symbol,time,mechanism,restriction,n
0,SWI1,5.0,ZEV,P,3
1,SWI1,15.0,ZEV,P,3
2,SWI1,90.0,ZEV,P,3
3,SWI1,45.0,ZEV,P,3
4,SWI1,10.0,ZEV,P,3
5,SWI1,20.0,ZEV,P,3
6,SWI1,30.0,ZEV,P,3
7,SWI1,0.0,ZEV,P,3
8,RDS2,45.0,ZEV,P,2
9,Z3EV,30.0,ZEV,P,2


In [14]:
# SWI1 has 3 samples at time=20, mechanism=ZEV. Let's look at just those samples
vdb.query("""
    SELECT *
    FROM hackett_meta
    WHERE regulator_symbol = 'SWI1'
      AND time = 20
      AND mechanism = 'ZEV'
""")

,sample_id,date,mechanism,regulator_locus_tag,regulator_symbol,restriction,strain,time,carbon_source,temperature_celsius
0,1636,20161117,ZEV,YPL016W,SWI1,P,SMY2266c,20.0,glucose,30.0
1,1628,20161117,ZEV,YPL016W,SWI1,P,SMY2266b,20.0,glucose,30.0
2,1620,20161117,ZEV,YPL016W,SWI1,P,SMY2266a,20.0,glucose,30.0


In [15]:
# In this case, there are three strains with otherwise the same experimental conditions.
# Rather than trying to choose among these right now, we might just want to get a
# unique list of the regulators with replicates in order to exclude them from an
# analysis that doesn't expect replicates.
replicated_hackett_regulators = vdb.query("""
    SELECT DISTINCT regulator_symbol
    FROM hackett_meta
    GROUP BY regulator_symbol, time, mechanism, restriction
    HAVING COUNT(*) > 1
""").regulator_symbol.tolist()
print(replicated_hackett_regulators)

['GCN4', 'MAC1', 'RDS2', 'Z3EV', 'SWI1']


In [16]:
# GEV is another "regulator" we want to exclude
replicated_hackett_regulators.append("GEV")
print(replicated_hackett_regulators)

['GCN4', 'MAC1', 'RDS2', 'Z3EV', 'SWI1', 'GEV']


In [17]:
vdb.query("SELECT * FROM dto_expanded")

,binding_id,perturbation_id,binding_rank_threshold,perturbation_rank_threshold,binding_set_size,perturbation_set_size,dto_fdr,dto_empirical_pvalue,binding_repo_dataset,perturbation_repo_dataset,binding_id_id,binding_id_source,perturbation_id_id,perturbation_id_source
0,BrentLab/harbison_2004;harbison_2004;105,BrentLab/hughes_2006;overexpression;10,11.0,206.0,12.0,206.0,0.041293,0.017,harbison_2004-harbison_2004,hughes_2006-overexpression,105,harbison,10,BrentLab/hughes_2006;overexpression
1,BrentLab/harbison_2004;harbison_2004;108,BrentLab/hughes_2006;overexpression;11,60.0,67.0,60.0,67.0,0.054284,0.000,harbison_2004-harbison_2004,hughes_2006-overexpression,108,harbison,11,BrentLab/hughes_2006;overexpression
2,BrentLab/harbison_2004;harbison_2004;109,BrentLab/hughes_2006;overexpression;11,27.0,1265.0,27.0,1265.0,0.123214,0.057,harbison_2004-harbison_2004,hughes_2006-overexpression,109,harbison,11,BrentLab/hughes_2006;overexpression
3,BrentLab/harbison_2004;harbison_2004;112,BrentLab/hughes_2006;overexpression;12,532.0,1093.0,532.0,1093.0,0.436305,0.092,harbison_2004-harbison_2004,hughes_2006-overexpression,112,harbison,12,BrentLab/hughes_2006;overexpression
4,BrentLab/harbison_2004;harbison_2004;113,BrentLab/hughes_2006;overexpression;12,10.0,556.0,10.0,556.0,0.017567,0.002,harbison_2004-harbison_2004,hughes_2006-overexpression,113,harbison,12,BrentLab/hughes_2006;overexpression
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29799,BrentLab/callingcards;annotated_features_combi...,BrentLab/kemmeren_2014;kemmeren_2014;784,154.0,905.0,154.0,905.0,0.090665,0.000,callingcards-annotated_features_combined,kemmeren_2014-kemmeren_2014,724-692-688,BrentLab/callingcards;annotated_features_combined,784,kemmeren
29800,BrentLab/callingcards;annotated_features_combi...,BrentLab/kemmeren_2014;kemmeren_2014;666,215.0,108.0,215.0,108.0,0.075036,0.005,callingcards-annotated_features_combined,kemmeren_2014-kemmeren_2014,725-435-395,BrentLab/callingcards;annotated_features_combined,666,kemmeren
29801,BrentLab/callingcards;annotated_features_combi...,BrentLab/kemmeren_2014;kemmeren_2014;271,221.0,925.0,221.0,925.0,0.403484,0.126,callingcards-annotated_features_combined,kemmeren_2014-kemmeren_2014,726-445-424,BrentLab/callingcards;annotated_features_combined,271,kemmeren
29802,BrentLab/callingcards;annotated_features_combi...,BrentLab/kemmeren_2014;kemmeren_2014;1077,281.0,73.0,283.0,77.0,0.095948,0.174,callingcards-annotated_features_combined,kemmeren_2014-kemmeren_2014,79-33,BrentLab/callingcards;annotated_features_combined,1077,kemmeren


In [18]:
# We can remove those regulators from our query using a parameterized query
hackett_harbison_dto = vdb.query("""
SELECT h.sample_id, h.regulator_symbol, h.time, h.mechanism,
       dto.*
FROM hackett_meta h
LEFT JOIN (
    SELECT *
    FROM dto_expanded
) AS dto
ON CAST(h.sample_id AS VARCHAR) = dto.perturbation_id_id
WHERE h.regulator_symbol NOT IN $replicated_hacket_regulators
    AND h.mechanism = 'ZEV'
    AND h.restriction = 'P'
    AND h.time = 15
ORDER BY h.regulator_symbol, h.time, h.mechanism
""",
    replicated_hacket_regulators=replicated_hackett_regulators
)
print(hackett_harbison_dto.head())

   sample_id regulator_symbol  time mechanism  \
0        448             ACA1  15.0       ZEV   
1        448             ACA1  15.0       ZEV   
2        448             ACA1  15.0       ZEV   
3        448             ACA1  15.0       ZEV   
4        448             ACA1  15.0       ZEV   

                                     binding_id  \
0  BrentLab/callingcards;annotated_features;156   
1  BrentLab/callingcards;annotated_features;146   
2  BrentLab/callingcards;annotated_features;126   
3  BrentLab/callingcards;annotated_features;803   
4  BrentLab/callingcards;annotated_features;390   

                          perturbation_id  binding_rank_threshold  \
0  BrentLab/hackett_2020;hackett_2020;448                   374.0   
1  BrentLab/hackett_2020;hackett_2020;448                   452.0   
2  BrentLab/hackett_2020;hackett_2020;448                   437.0   
3  BrentLab/hackett_2020;hackett_2020;448                   110.0   
4  BrentLab/hackett_2020;hackett_2020;448            

In [19]:
# Clean up temp file
temp_config.unlink(missing_ok=True)